In [1]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    pipeline
)
from huggingface_hub import login
from peft import LoraConfig, get_peft_model

c:\Users\Kostya\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.2)
  from scipy.sparse import csr_matrix, issparse


In [ ]:
HF_TOKEN = ""
login(token=HF_TOKEN)

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
LOCAL_MODEL_DIR = "./qwen2.5-0.5b"

os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)


In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
    cache_dir=LOCAL_MODEL_DIR
)

config = AutoConfig.from_pretrained(
    BASE_MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
    cache_dir=LOCAL_MODEL_DIR
)

model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    config=config,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.train()

`torch_dtype` is deprecated! Use `dtype` instead!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_features=

In [4]:
MAX_JOKES = 10000
jokes = []
with open("full_jokes.txt", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        line = line.strip()
        if line:
            jokes.append(line)
        if i + 1 >= MAX_JOKES*2:
            break

print(f"Используем {len(jokes)} анекдотов для дообучения")

Используем 10000 анекдотов для дообучения


In [15]:
def make_example(joke: str):
    words = joke.split()
    if len(words) < 6:
        return None

    start = " ".join(words[:4])
    continuation = joke[len(start):].strip()

    prompt = (
        "Продолжи анекдот на русском языке. "
        "Сделай его коротким и смешным.\n\n"
        f"{start}"
    )

    text = (
        f"<|user|>\n{prompt}\n"
        f"<|assistant|>\n{continuation}"
    )

    return {"text": text}

examples = []
for j in jokes:
    ex = make_example(j)
    if ex:
        examples.append(ex)

dataset = Dataset.from_list(examples)

In [55]:
training_args = TrainingArguments(
    output_dir="./qwen_anekdot_sft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=5e-5,
    bf16=True,
    logging_steps=20,
    save_steps=500,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

trainer.train()

trainer.save_model("./qwen_anekdot_sft")
tokenizer.save_pretrained("./qwen_anekdot_sft")

C:\Users\Kostya\AppData\Local\Temp\ipykernel_28028\1738722917.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,2.876700
40,2.304500
60,1.822600
80,1.460900
100,1.304800
120,1.283300
140,1.258500
160,1.272900
180,1.235600
200,1.235500


c:\Users\Kostya\AppData\Local\Programs\Python\Python312\Lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in ./qwen2.5-0.5b - will assume that the vocabulary was not modified.
  warnings.warn(
c:\Users\Kostya\AppData\Local\Programs\Python\Python312\Lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in ./qwen2.5-0.5b - will assume that the vocabulary was not modified.
  warnings.warn(


('./qwen_anekdot_sft\\tokenizer_config.json',
 './qwen_anekdot_sft\\special_tokens_map.json',
 './qwen_anekdot_sft\\chat_template.jinja',
 './qwen_anekdot_sft\\vocab.json',
 './qwen_anekdot_sft\\merges.txt',
 './qwen_anekdot_sft\\added_tokens.json',
 './qwen_anekdot_sft\\tokenizer.json')

In [20]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
model.to(device)

starts_with_index = {
    1: "Идёт мужик по лесу",
    2: "Встречаются два друга",
    3: "Приходит мужик в бар",
    4: "Жена говорит мужу",
    5: "Приходят альфа, бета и гамма в бар",
    6: "Идёт медведь по лесу",
    7: "Приходит мужик к врачу",
    8: "Встречаются русский, американец и немец",
    9: "Идёт по улице девушка",
    10: "Приходит мужик в магазин",
    11: "Еще сто лет назад",
    12: "Встречаются Вовочка и Петька",
    13: "Идёт по лесу охотник",
    14: "Я хорошо готовлю, стираю и убираю в квартире",
    15: "Жена спрашивает у мужа",
    16: "Сидят в баре два друга",
    17: "Идёт по пустыне караван",
    18: "Приходит мужик в аптеку",
    19: "Встречаются два программиста",
    20: "- Послушайте, у этого парня в резюме",
    21: "Приходит мужик в банк",
    22: "Сидят на скамейке два пенсионера",
    23: "Идёт по лесу грибник",
    24: "Приходит мужик в ресторан",
    25: "- Я дочитал учебник по теории вероятности",
    26: "Идёт по улице студент",
    27: "Заходит студент в кофейню",
    28: "Сидят в очереди два человека",
    29: "Идёт по лесу шаман",
    30: "Приходит мужик в библиотеку",
    31: "Идёт по улице кот",
    32: "Встречаются два математика",
    33: "Приходит программист в бар",
    34: "Сидит кот на клавиатуре",
    35: "Доказывает теорему математик",
    36: "Пишет код программист",
    37: "Спрашивает LLM у пользователя",
    38: "Встречаются feature engineer и data scientist",
    39: "Решает уравнение студент",
    40: "Вышел новый альбом Оксимирона",
    41: "Говорит кот хозяину",
    42: "Простой способ остудить чай",
    43: "Доказывает математик теорему",
    44: "Спрашивает математик у кота",
    45: "Пишет промпт для LLM",
    46: "Сидит кот перед монитором",
    47: "Объясняет математик программисту",
    48: "Наняли команду 40 программистов",
    49: "Идёт по крыше кот",
    50: "В статье было написано",
    51: "Спрашивает кот у математика",
    52: "Думает программист о баге",
    53: "Общается пользователь с LLM",
    54: "Сидит кот на книге по алгоритмам",
    55: "Пишет программист тесты",
    56: "Решает LLM задачу по математике",
    57: "Я прочитал книгу Пелевина",
    58: "Встречаются два кота",
    59: "Доказывает программист, что кот — это баг",
    60: "Как часто девушки думают о",
    61: "Примерно двадцать лет назад",
    62: "Классический ML",
    63: "Узнал сегодня забавный факт",
    64: "Я хотел быть самим собой, обычным пацаном",
    65: "За окном шумит Сургут",
    66: "Вообще я люблю только две вещи:",
    67: "Хороший русский рэп",
    68: "Одна бессмысленная ночь у телефона",
    69: "В России запретили",
    70: "Из характеристики:",
    76: "Встречаются overfitting и underfitting",
    80: "Идёт по дому кошка",
    81: "Я из тех людей",
    82: "- Я нормальный.",
    83: "Есть только одна система:"
}

generated = []

for idx, start in starts_with_index.items():
    prompt = f"Продолжи анекдот на русском языке. Сделай его коротким и смешным.\n\n{start}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.9,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    continuation = text[len(prompt):].strip().replace("\n", " ")
    generated.append(f"{idx} {continuation}")


with open("anekdots.txt", "w", encoding="utf-8") as f:
    for line in generated:
        f.write(line + "\n")

print(f"Сгенерировано {len(generated)} анекдотов. Файл anekdots.txt готов.")


Сгенерировано 75 анекдотов. Файл anekdots.txt готов.
